# M10.7 · GPU'da 2B konvolüsyon ve matris çarpımı — karolar, halo, sabit bellek, im2col

Paralel programlama alıştırmalarının GPU yarısı: uint8 görüntüde fp32 2B konvolüsyon (3×3 … 21×21) ve fp32 matris çarpımı. Konvolüsyonda girdi karosu = çıktı karosu + (K − 1) halo; filtre sabit bellekte (warp aynı katsayıyı okur: yayın), karo stratejileri ve veri yeniden kullanım oranı; im2col ile GEMM'e dönüştürme, örtük GEMM, çok kanallı konvolüsyon, FFT ve Winograd. Matmul'da paylaşımlı bellekli karolar (global okumalar T kat azalır), sınır kontrolü, yazmaç karolaması/kabalaştırma ve aritmetik yoğunluk, L2 için program sırası (gruplama). Saf Python'da halo'lu karo simülasyonu ve okuma sayımı, NumPy'da im2col/çok kanallı/FFT konvolüsyon, Triton'da karolu konvolüsyon ve gruplanmış matmul, sonda konvolüsyon yöntemlerinin (doğrudan, unfold + matmul, FFT) çekirdek boyutuna göre kesişimi.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-7-gpu-conv.html](https://m-gonul.github.io/pytorch-lab/m10-7-gpu-conv.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_7_gpu_conv.py
CHECKS = {
 "b": "import random\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rng = random.Random(0)\n\n    def conv():\n        ensure(need(ns, \"conv_tile_loads\")(16, 5) == (400, 1.5625), \"el hesabı\")\n        img = [[rng.randint(0, 255) for _ in range(40)] for _ in range(36)]\n        rows = []\n        for K in (3, 7, 11):\n            k = [[rng.random() for _ in range(K)] for _ in range(K)]\n            d, rd = need(ns, \"direct_conv\")(img, k)\n            t, rt = need(ns, \"tiled_conv\")(img, k, 8)\n            close(np.array(t), np.array(d), atol=1e-9, what=f\"K = {K}: karolu = doğrudan\")\n            rows.append([K, rd, rt, f\"{rd / rt:.1f}\", f\"{K * K / ns['conv_tile_loads'](8, K)[1]:.1f}\"])\n        R_.table(\"36×40 görüntü, 8×8 çıktı karoları: global okuma\", [\"K\", \"doğrudan\", \"karolu\", \"oran (ölçülen)\", \"oran (iç karo formülü)\"], rows,\n                 note=\"Ölçülen oran formülden biraz düşük: kenar karolarında görüntü dışı (sıfır) halo okunmuyor ve kenar çıktıları da daha az okuyor. Büyük K'de karolama vazgeçilmez.\")\n    R_.test(\"karolu konvolüsyon\", conv)\n\n    def mm():\n        A = [[rng.random() for _ in range(37)] for _ in range(29)]\n        B = [[rng.random() for _ in range(23)] for _ in range(37)]\n        for T in (1, 4, 8):\n            C, reads = need(ns, \"tiled_matmul\")(A, B, T)\n            close(np.array(C), np.array(A) @ np.array(B), what=f\"T = {T}\")\n        n = 32\n        A2 = [[1.0] * n for _ in range(n)]\n        r1 = ns[\"tiled_matmul\"](A2, A2, 1)[1]\n        r8 = ns[\"tiled_matmul\"](A2, A2, 8)[1]\n        ensure(r1 == 2 * n ** 3 and r8 == 2 * n ** 3 // 8, \"global okuma 2n³/T\")\n    R_.test(\"karolu matmul\", mm)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef direct(img, k):\n    K = k.shape[0]\n    r = K // 2\n    f = np.pad(img.astype(np.float64), r)\n    H, W = img.shape\n    return np.array([[(f[y:y + K, x:x + K] * k).sum() for x in range(W)] for y in range(H)])\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"im2col\", \"conv_im2col\", \"conv_multi\", \"conv_fft\", \"tile_intensity\")])\n    rng = np.random.default_rng(0)\n\n    def t():\n        img = rng.integers(0, 256, (30, 34)).astype(np.float32)\n        for K in (3, 7):\n            k = rng.random((K, K)).astype(np.float32)\n            ref = direct(img, k)\n            ensure(need(ns, \"im2col\")(img, K).shape == (30 * 34, K * K), \"im2col şekli\")\n            close(need(ns, \"conv_im2col\")(img, k), ref, rtol=1e-5, what=f\"im2col K = {K}\")\n            close(need(ns, \"conv_fft\")(img, k), ref, rtol=1e-6, atol=1e-6, what=f\"FFT K = {K}\")\n        x = rng.normal(size=(3, 12, 13)).astype(np.float32)\n        w = rng.normal(size=(5, 3, 3, 3)).astype(np.float32)\n        ref = np.stack([sum(direct(x[c], w[o, c]) for c in range(3)) for o in range(5)])\n        close(need(ns, \"conv_multi\")(x, w), ref, atol=1e-4, what=\"çok kanallı konvolüsyon = GEMM\")\n        close(need(ns, \"tile_intensity\")(32, 32, 32), 8.0, what=\"yoğunluk\")\n        big = rng.integers(0, 256, (256, 256)).astype(np.float32)\n        rows = []\n        for K in (3, 11, 31):\n            k = rng.random((K, K)).astype(np.float32)\n            rows.append([K, fmt_time(bench(lambda: ns[\"conv_im2col\"](big, k), repeat=2, number=1)), fmt_time(bench(lambda: ns[\"conv_fft\"](big, k), repeat=2, number=1))])\n        R_.table(\"256×256: im2col + matmul vs FFT\", [\"K\", \"im2col\", \"FFT\"], rows,\n                 note=\"im2col maliyeti K² ile büyür; FFT neredeyse K'den bağımsız. Derin ağlardaki 3×3'ler için GEMM/Winograd, sinyal işlemedeki büyük çekirdekler için FFT.\")\n    R_.test(\"konvolüsyon = matmul\", t)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef ref_conv(img, k):\n    K = k.shape[0]\n    acc = F.conv2d(img.float()[None, None], k.float()[None, None], padding=K // 2)[0, 0]\n    return torch.clamp((acc + 0.5).floor(), 0, 255).to(torch.uint8)\n\n\ndef check(ns, R_):\n    dev = ns.get(\"DEVICE\", \"cpu\")\n    g = torch.Generator().manual_seed(0)\n\n    def cv():\n        img = torch.randint(0, 256, (37, 45), dtype=torch.uint8, generator=g)\n        for K in (3, 7, 11):\n            gk = torch.exp(-((torch.arange(K) - K // 2).float() ** 2) / (K / 2) ** 2)\n            k = torch.outer(gk, gk)\n            k = k / k.sum()\n            out = need(ns, \"conv2d\")(img.to(dev), k).cpu()\n            d = (out.int() - ref_conv(img, k).int()).abs()\n            ensure(out.dtype == torch.uint8 and int(d.max()) <= 1 and float((d == 0).double().mean()) > 0.99, f\"K = {K}\")\n        sharp = torch.tensor([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=torch.float32)\n        d = (ns[\"conv2d\"](img.to(dev), sharp, 8, 16).cpu().int() - ref_conv(img, sharp).int()).abs()\n        ensure(int(d.max()) <= 1, \"keskinleştirme: [0, 255]'e kırpma, farklı karo boyutu\")\n    R_.test(\"Triton konvolüsyon (uint8 → fp32 → uint8)\", cv)\n\n    def mm():\n        for (M, K, N) in ((64, 64, 64), (50, 70, 33), (1, 5, 3), (130, 40, 70)):\n            a = torch.randn(M, K, generator=g).to(dev)\n            b = torch.randn(K, N, generator=g).to(dev)\n            close(need(ns, \"matmul\")(a, b).cpu().numpy(), (a @ b).cpu().numpy(), atol=1e-3, what=f\"{M}×{K} @ {K}×{N}\")\n        ai = torch.randint(-50, 50, (40, 30), generator=g, dtype=torch.int32)\n        bi = torch.randint(-50, 50, (30, 20), generator=g, dtype=torch.int32)\n        close(ns[\"matmul\"](ai.to(dev), bi.to(dev)).cpu().numpy(), (ai.float() @ bi.float()).numpy(), atol=1e-3, what=\"int32 girdi, fp32 hesap\")\n        R_.note(\"Gruplanmış sıra: ardışık programlar GROUP_M satır bloğunu paylaşır; GPU'da aynı anda çalışan programlar A'nın aynı satırlarını ve B'nin aynı sütunlarını L2'den yeniden kullanır (Triton öğreticisinde %10'a varan kazanç).\")\n    R_.test(\"Triton matmul\", mm)\n",
 "e": "import torch\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        g = torch.Generator().manual_seed(1)\n        img = torch.rand(40, 50, generator=g)\n        k = torch.rand(7, 7, generator=g)\n        ref = need(ns, \"conv_direct\")(img, k)\n        close(need(ns, \"conv_unfold\")(img, k).numpy(), ref.numpy(), atol=1e-4, what=\"unfold + matmul\")\n        close(need(ns, \"conv_fft\")(img, k).numpy(), ref.numpy(), atol=1e-3, what=\"FFT\")\n        res = need(ns, \"crossover\")(256, [3, 7, 15, 31, 51])\n        ensure(len(res) == 5 and all(r[4] < 1e-4 for r in res), \"üç yöntem aynı sonucu vermeli\")\n        ensure(res[-1][3] < res[-1][1], \"K = 51'de FFT doğrudan konvolüsyondan hızlı olmalı\")\n        R_.table(\"256×256, tek kanal (CPU)\", [\"K\", \"F.conv2d\", \"unfold + matmul\", \"FFT\", \"göreli fark\"], [[K, fmt_time(a), fmt_time(b), fmt_time(c), f\"{d:.0e}\"] for K, a, b, c, d in res],\n                 note=\"Doğrudan ve im2col yöntemlerin maliyeti K² ile, FFT'ninki neredeyse sabit. Kesişim noktası donanıma ve kütüphaneye bağlı; cuDNN her katman için birkaç algoritmayı deneyip en hızlısını seçer (torch.backends.cudnn.benchmark = True).\")\n    R_.test(\"yöntem kesişimi\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"conv1d_winograd\"))\n\n    def t():\n        d, g = [1.0, 2, 3, 4], [1.0, 0, -1]\n        close(np.array(need(ns, \"winograd_f23\")(d, g)), np.array([1 - 3, 2 - 4.0]), what=\"el örneği\")\n        rng = np.random.default_rng(0)\n        x = rng.normal(size=102)\n        g = rng.normal(size=3)\n        ref = np.correlate(x, g, mode=\"valid\")\n        close(need(ns, \"conv1d_winograd\")(x, g), ref, atol=1e-12, what=\"1B geçerli korelasyon\")\n        R_.note(\"Çıktı çifti başına 4 çarpma (6 yerine); filtre dönüşümü (g₀+g₁+g₂)/2 vb. bir kez önceden hesaplanır, girdi/çıktı dönüşümleri yalnız toplama. 2B F(2×2, 3×3): 16 çarpma vs 36 (2.25×).\")\n    R_.test(\"Winograd\", t)\n",
 "x2": "from labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        s = need(ns, \"config_stats\")(32, 32, 32, 1)\n        close(s[\"ai_global\"], 8.0, what=\"karo yoğunluğu\")\n        ensure(s[\"threads\"] == 1024 and s[\"ai_smem\"] == 0.25, \"r = 1: her FMA iki paylaşımlı okuma\")\n        cands = [(BM, BN, BK, r) for BM in (32, 64, 128, 256) for BN in (32, 64, 128, 256) for BK in (8, 16, 32) for r in (1, 2, 4, 8)]\n        best = need(ns, \"best_config\")(cands)\n        ensure(best is not None and need(ns, \"config_stats\")(*best)[\"smem_bytes\"] <= 48 * 1024, \"sınırlar içinde\")\n        rows = []\n        for c in [(32, 32, 32, 1), (64, 64, 16, 4), (128, 128, 8, 8), best]:\n            st = ns[\"config_stats\"](*c)\n            rows.append([str(c), st[\"threads\"], st[\"smem_bytes\"], st[\"regs_acc\"], f\"{st['ai_global']:.1f}\", f\"{st['ai_smem']:.2f}\"])\n        R_.table(\"Matmul blok yapılandırmaları (fp32)\", [\"(BM, BN, BK, r)\", \"thread\", \"smem B\", \"biriktirici/thread\", \"FLOP/B (global)\", \"FLOP/B (paylaşımlı)\"], rows,\n                 note=\"Paylaşımlı bellek de bir darboğazdır: r = 1'de her FMA için iki paylaşımlı okuma. r × r yazmaç karosu bunu r kat azaltır — cuBLAS benzeri çekirdeklerin (Boehm'in yazısı) asıl sırrı.\")\n    R_.test(\"yapılandırma uzayı\", t)\n",
 "x3": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for K in (3, 7, 21, 51):\n            a = need(ns, \"traffic_2d\")(1024, 1024, K, 16)\n            b = need(ns, \"traffic_separable\")(1024, 1024, K, 16)\n            rows.append([K, f\"{(a[0] + a[1]) / 2 ** 20:.1f}\", f\"{a[2] / 1e6:.0f}\", f\"{(b[0] + b[1]) / 2 ** 20:.1f}\", f\"{b[2] / 1e6:.0f}\"])\n        ensure(float(rows[0][1]) < float(rows[0][3]), \"küçük K'de tek geçiş daha az trafik\")\n        R_.table(\"1024², uint8 girdi/çıktı, ayrılabilir ara görüntü fp32 (bayt)\", [\"K\", \"2B trafik (MiB)\", \"2B MAC (M)\", \"ayrılabilir trafik (MiB)\", \"ayrılabilir MAC (M)\"], rows,\n                 note=\"Ayrılabilir sürüm MAC'leri K/2 kat azaltır ama ara görüntü global bellekten bir kez gidip gelir (fp32 olarak 8 MiB). Küçük K'de çekirdek bellek-bağımlıdır ve tek geçiş kazanır; büyük K'de hesap kazancı baskındır. İki geçişi tek çekirdekte paylaşımlı bellekle birleştirmek (füzyon) ikisinin en iyisini verir.\")\n    R_.test(\"trafik modeli\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_7_gpu_conv as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Alıştırma 4 (BİL 455/533): **CUDA 2B konvolüsyon** — girdi uint8, katsayılar fp32, hesap fp32, çıktı uint8; filtre 3×3, 7×7, 11×11, 21×21; çok thread'li C++ sürümüyle (M10.3) farklı görüntü boyutlarında karşılaştır. **CUDA matris çarpımı** — int32 ya da fp32 girdi, fp32 hesap; doğruluğu bir lineer cebir kütüphanesiyle denetle; C++ thread'leriyle karşılaştır.

**Konvolüsyon karoları:** her blok T×T çıktı üretir ama $(T+K-1)^2$ girdi okur (halo). Strateji 1: blok = girdi karosu kadar thread, hepsi yükler, yalnız içtekiler hesaplar (boşta thread). Strateji 2: blok = çıktı karosu, halo'yu birden çok yüklemeyle getir. Strateji 3: yalnız karo içini paylaşımlı belleğe al, halo'yu L1/L2 önbelleğine bırak. Yeniden kullanım: doğrudan sürüm her çıktı için $K^2$ okur, karolu $(T+K-1)^2/T^2$ — büyük K'de kazanç çok büyük. **Filtre sabit bellekte**: tüm warp aynı katsayıyı okur → yayın. Sınırda sıfır dolgu (hayalet hücreler) okunmaz.

**Konvolüsyonu GEMM'e çevirmek:** **im2col** her çıktı konumunun $C_{in}K^2$'lik pencere vektörünü bir satıra yazar → $(C_{out}\times C_{in}K^2)\cdot(C_{in}K^2\times HW)$ matmul. Bellek: girdi $K^2$ kat çoğalır; **örtük GEMM** satırları yükleme sırasında üretir. Alternatifler: **FFT** (büyük K'de $O(HW\log HW)$), **Winograd** (küçük K'de daha az çarpma).

**Karolu matmul:** blok bir $T\times T$ C karosundan sorumlu; her fazda A ve B'nin birer $T\times T$ karosunu paylaşımlı belleğe yükler, `__syncthreads`, T çarp-topla, tekrar. Global okuma $2n^3\to2n^3/T$; yoğunluk ~ $T/4$ FLOP/B (fp32). Kenarlarda sınır kontrolü (0 doldur). **Yazmaç karolaması / kabalaştırma:** her thread çıktıların bir $r\times r$ alt karosunu tutar → paylaşımlı bellekten okunan her değer r kez kullanılır. **Gruplanmış sıra:** program kimliklerini sütun gruplarında dolaştırmak, L2'de A ve B karolarının yeniden kullanımını artırır.

## 1 · Elle

K = 5: $(16+4)^2=400$, $400/256=1.5625$ okuma/çıktı, $25/1.5625=16$. K = 21: $36^2/256=5.06$, $441/5.06=87.1$.

Matmul fazı: $2T^3$ FLOP, A ve B karoları $2T^2\cdot4$ bayt → $T/4$. im2col: her piksel 441 kez yazılır: $512^2\cdot441\cdot4$ B = 441 MiB (görüntünün kendisi 0.25 MiB uint8). Sabit bellek: 16 384 float → 128 × 128.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Karolu konvolüsyon, çıktı karosu 16 × 16**

- `in5`: K = 5: girdi karosu eleman sayısı
- `r5`: K = 5: doğrudan (K² okuma) / karolu (girdi karosu / 256) oranı
- `r21`: K = 21 için aynı oran

**Matmul ve im2col**

- `ai`: fp32 karo T = 32: bir faz için FLOP/bayt ($2T^3/(2T^2\cdot4)$)
- `im2col`: 512 × 512 tek kanal, K = 21: im2col matrisinin boyutu (MiB, fp32)
- `cmem`: 64 KB sabit belleğe sığan en büyük kare fp32 filtre kenarı

In [ ]:
cevaplar = dict(
    in5=None,
    r5=None,
    r21=None,
    ai=None,
    im2col=None,
    cmem=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"in5":[400,0],"r5":[16,0],"r21":[87.11111111111111,0.000001],"ai":[8,0],"im2col":[441,0],"cmem":[128,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, okuma sayan simülasyonlar (sıfır dolgu: görüntü dışı okunmaz).

- `conv_tile_loads(T, K)` → `(girdi karosu eleman sayısı, çıktı başına okuma)`.
- `direct_conv(img, k)` → `(çıktı, global okuma)` — her çıktı kendi sınır içi komşularını okur; çıktı float (yuvarlamasız).
- `tiled_conv(img, k, T)` → `(çıktı, global okuma)` — her T×T çıktı karosu için halo'lu girdi karosunu (sınır içi kısmını) bir kez "paylaşımlı" listeye kopyala, çıktıları oradan hesapla.
- `tiled_matmul(A, B, T)` → `(C, global okuma)` — her C karosu için her fazda A ve B karolarını (sınır içi elemanları) bir kez oku.

In [ ]:
%%writefile stage_b.py
def conv_tile_loads(T, K):
    ...


def direct_conv(img, k):
    ...


def tiled_conv(img, k, T):
    ...


def tiled_matmul(A, B, T):
    ...


print(conv_tile_loads(16, 5))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz.

- `im2col(img, K)` → `(H·W, K²)` (sıfır dolgu, `sliding_window_view`).
- `conv_im2col(img, k)` — im2col @ k.ravel() (fp32).
- `conv_multi(x, w)` — `x (C_in, H, W)`, `w (C_out, C_in, K, K)` → `(C_out, H, W)`: `w.reshape(C_out, -1) @ cols`, cols `(C_in K², H W)`.
- `conv_fft(img, k)` — korelasyon: görüntü ve (ters çevrilmiş) çekirdek $(H+K-1)\times(W+K-1)$'e sıfır dolgu, `rfft2`, çarp, `irfft2`, ortadaki H×W.
- `tile_intensity(BM, BN, BK)` — bir faz için FLOP/bayt (fp32).

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def im2col(img, K):
    ...


def conv_im2col(img, k):
    ...


def conv_multi(x, w):
    ...


def conv_fft(img, k):
    ...


def tile_intensity(BM, BN, BK):
    ...


print(im2col(np.arange(9.0).reshape(3, 3), 3).shape)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Triton (GPU yoksa yorumlayıcı).

- `conv2d(img, k, BH=16, BW=16)` — `img` uint8 `(H, W)`, `k` fp32 `(K, K)`; her program BH×BW çıktı karosu; K² dokunuş için kaydırılmış girdi bloğunu (maskeli, dışı 0) yükle ve biriktir; çıktı $\min(255,\max(0,\lfloor acc+0.5\rfloor))$ uint8.
- `matmul(a, b, BM=32, BN=32, BK=32, GROUP_M=4)` — fp32 (int32 girdi fp32'ye çevrilir); program kimliğini **gruplanmış sırada** (GROUP_M satır bloğu × tüm sütunlar) eşle; `tl.dot`; kenar maskeleri.

In [ ]:
%%writefile stage_d.py
import os
import torch

if not torch.cuda.is_available():
    os.environ.setdefault("TRITON_INTERPRET", "1")
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def conv2d(img, k, BH=16, BW=16):
    ...


def matmul(a, b, BM=32, BN=32, BK=32, GROUP_M=4):
    ...


print(conv2d(torch.full((8, 8), 10, dtype=torch.uint8), torch.ones(3, 3) / 9))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

PyTorch (CPU), `img (H, W)` float32, `k (K, K)`.

- `conv_direct(img, k)` — `F.conv2d`, `padding=K//2`.
- `conv_unfold(img, k)` — `F.unfold` (im2col) + matmul.
- `conv_fft(img, k)` — `torch.fft.rfft2` ile.
- `crossover(H, Ks)` → her K için üç yöntemin süresi ve en büyük farkı.

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn.functional as F


def conv_direct(img, k):
    ...


def conv_unfold(img, k):
    ...


def conv_fft(img, k):
    ...


def crossover(H, Ks):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Winograd F(2, 3): 6 yerine 4 çarpma

**Winograd F(2, 3):** 4 girdi $d_0..d_3$, filtre $g_0..g_2$, 2 çıktı $y_0=d_0g_0+d_1g_1+d_2g_2$, $y_1=d_1g_0+d_2g_1+d_3g_2$:
$m_1=(d_0-d_2)g_0$, $m_2=(d_1+d_2)\frac{g_0+g_1+g_2}2$, $m_3=(d_2-d_1)\frac{g_0-g_1+g_2}2$, $m_4=(d_1-d_3)g_2$; $y_0=m_1+m_2+m_3$, $y_1=m_2-m_3-m_4$.

- `winograd_f23(d, g)` → `(y0, y1)`; `conv1d_winograd(x, g)` — 1B "geçerli" korelasyon, uzunluk L−2 (çift varsay), karolar hâlinde (NumPy, döngüsüz).

In [ ]:
%%writefile extra_x1.py
import numpy as np


def winograd_f23(d, g):
    ...


def conv1d_winograd(x, g):
    ...


print(winograd_f23([1.0, 2, 3, 4], [1.0, 0, -1]))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Yazmaç karolaması: blok şekli, yoğunluk ve kaynak sınırları

Blok başına (BM × BN çıktı, BK derinlik), thread başına (r × r) yazmaç karosu.

- `config_stats(BM, BN, BK, r)` → sözlük: `threads` = BM·BN/r², `smem_bytes` = 4(BM·BK + BK·BN), `regs_acc` = r² (thread başına biriktirici), `ai_global` = 2·BM·BN·BK / (4(BM·BK + BK·BN)), `ai_smem` = 2r²/(4·2r) (paylaşımlıdan okunan bayt başına FLOP).
- `best_config(candidates, smem_limit=48*1024, max_threads=1024, max_acc=64)` — sınırları sağlayanlar içinde `ai_global` en büyük (eşitlikte `ai_smem`).

In [ ]:
%%writefile extra_x2.py
def config_stats(BM, BN, BK, r):
    ...


def best_config(candidates, smem_limit=48 * 1024, max_threads=1024, max_acc=64):
    ...


print(config_stats(64, 64, 16, 4))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Ayrılabilir konvolüsyon GPU'da: iki geçişin trafik hesabı

H × W görüntü, K×K ayrılabilir filtre, T×T karolar.

- `traffic_2d(H, W, K, T)` → (global okuma, global yazma, MAC): tek geçişli karolu 2B.
- `traffic_separable(H, W, K, T)` — iki geçiş: yatay (karo T×T, halo yalnız yatayda), ara görüntü global belleğe (fp32) yazılır ve dikey geçişte okunur.

In [ ]:
%%writefile extra_x3.py
def traffic_2d(H, W, K, T):
    ...


def traffic_separable(H, W, K, T):
    ...


print(traffic_2d(1024, 1024, 21, 16), traffic_separable(1024, 1024, 21, 16))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Lavin & Gray (2016): Winograd ile hızlı konvolüsyon

[Lavin & Gray — Fast Algorithms for Convolutional Neural Networks (CVPR 2016)](https://arxiv.org/abs/1509.09308)

x1'deki 1B F(2,3)'ü 2B F(2×2, 3×3) ve F(4×4, 3×3)'e genişlet: G, Bᵀ, Aᵀ dönüşüm matrislerini kullanarak 4×4 (6×6) giriş karolarında eleman bazlı çarpım, sonra (çok kanallıda) kanal toplamı = karo başına GEMM. PyTorch ile VGG'nin 3×3 katmanlarında `F.conv2d` ile doğruluk ve FLOP karşılaştırması yap; fp16'da F(4×4)'ün sayısal hatasını ölç.


**Adımlar**

1. Dönüşüm matrislerini makalenin Ek'inden ya da Cook–Toom inşasından türet.

**Kabul ölçütleri**

- F(2×2,3×3) çarpma sayısı 2.25× az; F(4×4,3×3) 4× az.
- fp32'de göreli hata < 1e-5, fp16'da artışı raporlanmış.

## Makale ödevi: Chetlur ve ark. (2014): cuDNN ve örtük GEMM

[Chetlur ve ark. — cuDNN: Efficient Primitives for Deep Learning (2014)](https://arxiv.org/abs/1410.0759)

d aşamasındaki Triton matmul'u, girdi matrisinin satırlarını im2col indeksleriyle doğrudan görüntüden yükleyen bir **örtük GEMM** konvolüsyonuna dönüştür (N, C, H, W; çıkış kanalı = N boyutu). Ara im2col matrisi oluşturulmasın; doğruluğu `F.conv2d` ile, hızı (GPU'da) cuDNN ve açık im2col + matmul ile karşılaştır; makalenin Şekil 2–3'teki bellek/hız tartışmasını kendi ölçümlerinle tekrar et.


**Adımlar**

1. K-boyutu döngüsünde (c, u, v) üçlüsünü tek indeksten çöz.

**Kabul ölçütleri**

- Açık im2col'a göre bellek kullanımı K² kat az, hız aynı mertebede.

## Kaynaklar

- [Hwu, Kirk, El Hajj — Programming Massively Parallel Processors (4. baskı), Bölüm 5 ve 7](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — ©
- [Chetlur ve ark. — cuDNN: Efficient Primitives for Deep Learning (2014)](https://arxiv.org/abs/1410.0759) — arXiv
- [Lavin & Gray — Fast Algorithms for Convolutional Neural Networks (CVPR 2016)](https://arxiv.org/abs/1509.09308) — arXiv
- [Triton — Matrix Multiplication tutorial (L2 önbellek için gruplama)](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html) — MIT
- [Boehm — How to Optimize a CUDA Matmul Kernel for cuBLAS-like Performance](https://siboehm.com/articles/22/CUDA-MMM) — ©